# 05 · Further Practice: Second Example

**Lesson:** [Interactive Data Visualization Dashboard](https://programminghistorian.org/en/lessons/interactive-data-visualization-dashboard)

**Tutorial sections covered:**
- Dataset
- Planning the Dashboard
- Downloading the Data
- Building the Dashboard

## 🎯 Goal
Build a two-pie-chart dashboard of non-English newspapers by decade from Chronicling America data.

## Dataset & Planning the Dashboard

In [1]:
# Run the "Downloading the Data" cell below first if ../data/data_lang_asrow.csv doesn't exist yet.
import pandas as pd

lang_asrow = pd.read_csv("../data/data_lang_asrow.csv", encoding="utf-8", index_col=[0])
print(lang_asrow.shape)   # (languages, decades)
lang_asrow.head(10)

(92, 34)


,1690,1700,1710,1720,1730,1740,1750,1760,1770,1780,...,1930,1940,1950,1960,1970,1980,1990,2000,2010,2020
Albanian,0,0,0,0,0,0,0,0,0,0,...,3,4,4,4,4,4,5,3,3,3
Amharic,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,1,1
Arabic,3,3,3,3,3,3,3,3,3,3,...,43,44,44,43,45,52,63,58,58,58
Armenian,1,1,1,1,1,1,1,1,1,1,...,17,16,18,18,18,24,24,14,15,15
Asue Awyu,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Austronesian (Other),0,0,0,0,0,0,0,0,0,0,...,2,2,2,3,2,3,2,1,1,1
Basque,0,0,0,0,0,0,0,0,0,0,...,2,2,2,2,2,2,2,0,0,0
Belarussian,0,0,0,0,0,0,0,0,0,0,...,1,2,4,4,4,4,4,2,2,2
Bengali,1,1,1,1,1,1,1,1,1,1,...,6,6,6,6,6,9,11,11,11,11
Bosnian,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,2,2,2,2


In [2]:
# Which languages had the most newspapers in the most recent decade?
lang_asrow["2020"].sort_values(ascending=False).head(10)

English    19474
Spanish      990
Chinese      125
Russian      101
German        93
Polish        61
Arabic        58
Korean        53
Italian       49
French        40
Name: 2020, dtype: int64

### 📝 My notes
- English by far. Spanish is the next closest with almost a 20th of the size of English papers. 

## Downloading the Data

In [3]:
# The lesson's rq2-download.py scrapes Chronicling America for hours (and the site has
# since changed), so this downloads the finished CSV the lesson provides instead.
# Saved twice: ../data/ for this notebook, ../tutorial_app/ so Render can find it.
import requests

csv_url = "https://programminghistorian.org/assets/interactive-data-visualization-dashboard/data_lang_asrow.csv"
response = requests.get(csv_url)
response.raise_for_status()

for path in ["../data/data_lang_asrow.csv", "../tutorial_app/data_lang_asrow.csv"]:
    with open(path, "wb") as f:
        f.write(response.content)
    print("Saved", path)

Saved ../data/data_lang_asrow.csv
Saved ../tutorial_app/data_lang_asrow.csv


### 📝 My notes
- Pretty straightforward process. I like using the .. command. 

## Building the Dashboard

In [4]:
# Imports and data preparation
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px

lang_asrow = pd.read_csv("../data/data_lang_asrow.csv", encoding="utf-8", index_col=[0])
lang_asrow_noeng = lang_asrow.drop(['English'])
lang_asrow_noeng.columns = lang_asrow_noeng.columns.astype(str)

lang_asrow_noeng.head()

,1690,1700,1710,1720,1730,1740,1750,1760,1770,1780,...,1930,1940,1950,1960,1970,1980,1990,2000,2010,2020
Albanian,0,0,0,0,0,0,0,0,0,0,...,3,4,4,4,4,4,5,3,3,3
Amharic,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,1,1
Arabic,3,3,3,3,3,3,3,3,3,3,...,43,44,44,43,45,52,63,58,58,58
Armenian,1,1,1,1,1,1,1,1,1,1,...,17,16,18,18,18,24,24,14,15,15
Asue Awyu,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# App instance
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.LITERA])
server = app.server

In [6]:
# Frontend: title row, then two columns, each with a decade dropdown and a pie chart
app.layout = dbc.Container(
         [
        dbc.Row([ # row 1
            dbc.Col([html.H1('Top 10 Non-English Newspaper Languages in the U.S.')],
                    className="text-center mt-3 mb-1")
        ]
    ),
        dbc.Row([ # row 2
            dbc.Col( # left column
                dbc.Row([
                dbc.Label("Select a decade:", className="fw-bold"),
                dcc.Dropdown(id='years_left',
                        options=lang_asrow_noeng.columns,
                        value='1690', clearable=False
                        ),
            dcc.Graph(id="graph_left")
            ]),
                width=6
            ),
            dbc.Col( # right column
                dbc.Row([
                dbc.Label("Select a decade:", className="fw-bold"),
                dcc.Dropdown(id='years_right',
                        options=lang_asrow_noeng.columns,
                        value='2020', clearable=False
                ),
            dcc.Graph(id="graph_right")
            ]),
                width=6,
            ),
        ])
], fluid=True)

In [7]:
# Backend: when either dropdown changes, rebuild both pie charts (top 10 + "Others")
@app.callback(
    Output("graph_left", "figure"),
    Output("graph_right", "figure"),
    Input("years_left", "value"),
    Input("years_right", "value")
    )
def generate_chart(year_left, year_right):
    df_left = lang_asrow_noeng[year_left].sort_values(ascending = False).reset_index().head(10)

    df_left.loc[len(df_left)] = ["Others", lang_asrow_noeng[year_left].sort_values(ascending = False).reset_index().iloc[10:][year_left].sum()]

    df_left = df_left.rename(columns={"index": "Language", year_left: "Count"})

    fig_left = px.pie(df_left, values="Count", names="Language", hole=.6,
                color_discrete_sequence=px.colors.sequential.RdBu)

    df_right = lang_asrow_noeng[year_right].sort_values(ascending = False).reset_index().head(10)

    df_right.loc[len(df_right)] = ["Others", lang_asrow_noeng[year_right].sort_values(ascending = False).reset_index().iloc[10:][year_right].sum()]

    df_right = df_right.rename(columns={"index": "Language", year_right: "Count"})

    fig_right = px.pie(df_right, values="Count", names="Language", hole=.6,
                color_discrete_sequence=px.colors.sequential.RdBu)
    return fig_left, fig_right

In [8]:
# Test the dashboard inside the notebook (port 8051 so it doesn't clash with notebook 03's app)
app.run(jupyter_mode="inline", debug=True, port=8051)

### 📝 My notes
- Very cool to look at. I like the labels within the pie charts. It is cool to examine the settings that build the layout of the chart. 

## Export to `app-rq2.py`
Writes the standalone version Render runs (Start Command: `gunicorn app-rq2:server`).
It reads `data_lang_asrow.csv` from the `tutorial_app` folder, so run the download cell first.

In [9]:
%%writefile ../tutorial_app/app-rq2.py
# Standalone second-example dashboard for Render (exported from notebook 05).
# Run locally with:  python app-rq2.py   — on Render: gunicorn app-rq2:server

import os
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px

# --- Data: languages as rows, decades as columns ---
csv_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data_lang_asrow.csv")
lang_asrow = pd.read_csv(csv_path, encoding="utf-8", index_col=[0])
lang_asrow_noeng = lang_asrow.drop(['English'])
lang_asrow_noeng.columns = lang_asrow_noeng.columns.astype(str)

# --- App instance ---
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.LITERA])
server = app.server

# --- Frontend ---
app.layout = dbc.Container(
         [
        dbc.Row([ # row 1
            dbc.Col([html.H1('Top 10 Non-English Newspaper Languages in the U.S.')],
                    className="text-center mt-3 mb-1")
        ]
    ),
        dbc.Row([ # row 2
            dbc.Col( # left column
                dbc.Row([
                dbc.Label("Select a decade:", className="fw-bold"),
                dcc.Dropdown(id='years_left',
                        options=lang_asrow_noeng.columns,
                        value='1690', clearable=False
                        ),
            dcc.Graph(id="graph_left")
            ]),
                width=6
            ),
            dbc.Col( # right column
                dbc.Row([
                dbc.Label("Select a decade:", className="fw-bold"),
                dcc.Dropdown(id='years_right',
                        options=lang_asrow_noeng.columns,
                        value='2020', clearable=False
                ),
            dcc.Graph(id="graph_right")
            ]),
                width=6,
            ),
        ])
], fluid=True)

# --- Backend ---
@app.callback(
    Output("graph_left", "figure"),
    Output("graph_right", "figure"),
    Input("years_left", "value"),
    Input("years_right", "value")
    )
def generate_chart(year_left, year_right):
    df_left = lang_asrow_noeng[year_left].sort_values(ascending = False).reset_index().head(10)

    df_left.loc[len(df_left)] = ["Others", lang_asrow_noeng[year_left].sort_values(ascending = False).reset_index().iloc[10:][year_left].sum()]

    df_left = df_left.rename(columns={"index": "Language", year_left: "Count"})

    fig_left = px.pie(df_left, values="Count", names="Language", hole=.6,
                color_discrete_sequence=px.colors.sequential.RdBu)

    df_right = lang_asrow_noeng[year_right].sort_values(ascending = False).reset_index().head(10)

    df_right.loc[len(df_right)] = ["Others", lang_asrow_noeng[year_right].sort_values(ascending = False).reset_index().iloc[10:][year_right].sum()]

    df_right = df_right.rename(columns={"index": "Language", year_right: "Count"})

    fig_right = px.pie(df_right, values="Count", names="Language", hole=.6,
                color_discrete_sequence=px.colors.sequential.RdBu)
    return fig_left, fig_right

if __name__ == '__main__':
    app.run(debug=True)

Overwriting ../tutorial_app/app-rq2.py


## ✅ Takeaways
*What would I reuse for my own dashboard?*
- I would definitely take the pie charts. Also, the data wrangling was 